# 02 — Correlation with human scores (B-08; figures: B-09)

Reads `results/scores.csv` (from `scripts/compute_scores.py`) and writes tables to `results/tables/`.

**Direction:** LPIPS and DISTS are distances (lower = better), so `evaluate.table` negates them before correlating. MOS is higher = better.
**PLCC:** reported both raw and after the VQEG 5-parameter logistic fit (D-05).

In [1]:
import pandas as pd
from humper.data import REPO_ROOT
from humper.evaluate import table
from humper.metrics import METRICS

TAB = REPO_ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)
scores = pd.read_csv(REPO_ROOT / "results" / "scores.csv")
names = [m.name for m in METRICS]
assert scores[names].notna().all().all(), "scores.csv incomplete: rerun compute_scores.py"
print(len(scores), "pairs; device:", scores.device.unique())

10125 pairs; device: <ArrowStringArray>
['mps']
Length: 1, dtype: str


## Direction check
Raw SRCC vs MOS: positive for higher-is-better metrics, negative for distances. After alignment, all should be positive.

In [2]:
pd.DataFrame({
    "higher_is_better": {m.name: m.higher_is_better for m in METRICS},
    "raw_srcc_vs_mos": {m.name: scores[m.name].corr(scores.mos, method="spearman") for m in METRICS},
}).round(3)

,higher_is_better,raw_srcc_vs_mos
psnr,True,0.676
ssim,True,0.619
ms_ssim,True,0.826
lpips,False,-0.822
dists,False,-0.814


## Overall

In [3]:
overall = table(scores)
overall.to_csv(TAB / "overall.csv", index=False)
overall.round(3)

,subset,metric,srcc,krcc,plcc_raw,plcc,fit_ok,n
0,all,psnr,0.676,0.488,0.556,0.680,True,10125
1,all,ssim,0.619,0.447,0.576,0.628,True,10125
2,all,ms_ssim,0.826,0.635,0.680,0.823,True,10125
3,all,lpips,0.822,0.630,0.748,0.823,True,10125
4,all,dists,0.814,0.625,0.806,0.814,True,10125


## Per distortion group / type

In [4]:
by_group = table(scores, by="dist_group")
by_type = table(scores, by="dist_name")
by_group.to_csv(TAB / "by_group.csv", index=False)
by_type.to_csv(TAB / "by_type.csv", index=False)
by_group.pivot(index="dist_group", columns="metric", values="srcc")[names].round(3)

metric,psnr,ssim,ms_ssim,lpips,dists
dist_group,,,,,
blur,0.888,0.911,0.950,0.924,0.965
brightness,0.900,0.856,0.879,0.901,0.892
color,0.683,0.310,0.743,0.660,0.651
compression,0.827,0.855,0.918,0.903,0.917
noise,0.820,0.768,0.858,0.784,0.838
sharpness_contrast,0.872,0.866,0.923,0.935,0.918
spatial,0.681,0.386,0.713,0.786,0.632


In [5]:
srcc_type = by_type.pivot(index="dist_name", columns="metric", values="srcc")[names]
srcc_type.to_csv(TAB / "srcc_by_type.csv")
srcc_type.round(3).sort_values("lpips")

metric,psnr,ssim,ms_ssim,lpips,dists
dist_name,,,,,
color_block,0.518,0.586,0.551,0.567,0.591
non_eccentricity_patch,0.289,0.410,0.438,0.614,0.672
color_saturation_1,0.523,0.467,0.491,0.659,0.685
pixelate,0.421,0.444,0.483,0.718,0.784
color_shift,0.558,0.577,0.731,0.746,0.843
color_quantization,0.809,0.797,0.850,0.747,0.794
white_noise,0.853,0.852,0.858,0.757,0.820
mean_shift,0.759,0.679,0.721,0.771,0.783
impulse_noise,0.877,0.889,0.873,0.780,0.808


## Best metric per distortion type
Interpretation notes go in B-10 (humans).

In [6]:
srcc_type.idxmax(axis=1).value_counts()

dists      13
ms_ssim     6
lpips       4
ssim        1
psnr        1
Name: count, dtype: int64

## Sensitivity: pixel-identical pairs
320 KADID 'distorted' images are identical to their reference (4 per ref: gaussian/motion blur L1, saturation-2 L1, mean-shift L3). Every metric gives them a perfect score. Main results keep them (they are part of the dataset, D-18); this table checks how much they matter.

In [7]:
identical = scores.psnr >= 80
print(identical.sum(), "identical pairs:", scores[identical].dist_name.value_counts().to_dict())
excl = table(scores[~identical])
excl.to_csv(TAB / "overall_excl_identical.csv", index=False)
overall.set_index("metric")[["srcc", "plcc"]].join(excl.set_index("metric")[["srcc", "plcc"]], rsuffix="_excl").round(3)

320 identical pairs: {'gaussian_blur': 80, 'motion_blur': 80, 'color_saturation_2': 80, 'mean_shift': 80}


/var/folders/x0/r1z5zlqn04b2l8k0gt5shsv80000gn/T/ipykernel_32734/1493781188.py:5: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  overall.set_index("metric")[["srcc", "plcc"]].join(excl.set_index("metric")[["srcc", "plcc"]], rsuffix="_excl").round(3)


,srcc,plcc,srcc_excl,plcc_excl
metric,,,,
psnr,0.676,0.680,0.646,0.654
ssim,0.619,0.628,0.583,0.592
ms_ssim,0.826,0.823,0.810,0.809
lpips,0.822,0.823,0.807,0.809
dists,0.814,0.814,0.797,0.799
